# 구글플레이에 등록된 앱의 리뷰 추출

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

# 웹브라우저를 실행 할 때 사용할 옵션 - 사람인 것처럼 정보 입력
options = Options()
options.add_experimental_option("detach", True)
options.add_argument("start-maximized")
options.add_argument("Chrome/141.0.0.0")
options.add_argument("lang=ko_KR")


driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
    )
# 웹브라우저에서 url 열기
driver.get("https://play.google.com/store/apps/details?id=viva.republica.toss")

In [2]:
import time

In [ ]:
driver.execute_script(f"window.scrollTo({0}, {1400})")
time.sleep(2)

In [ ]:
# 평점 및 리뷰 옆의 -> 버튼 클릭
button = driver.find_element(By.CSS_SELECTOR, 'button[aria-label*="평점 및 리뷰 자세히 알아보기"]')
button.click()

In [ ]:
# 최신순으로 리뷰를 정렬하기 위해서 버튼 클릭
driver.find_element(By.ID, "sortBy_1").click()

In [ ]:
# 최신을 찾아 클릭
driver.find_element(By.CSS_SELECTOR, 'span[aria-label*="최신"]').click()

In [ ]:
# window를 스크롤하면 리뷰가 아닌 배경 페이지가 스크롤됨
# driver.execute_script(f"window.scrollTo({0}, {2400})")
# time.sleep(2)

In [ ]:
# 리뷰가 담긴 창을 찾아서 Javascript로 1000px씩 아래로 스크롤
driver.execute_script("document.querySelector('.fysCi.Vk3ZVd').scrollBy(0, 10000)")

In [ ]:
# 리뷰일
review_date = driver.find_element(By.CSS_SELECTOR, ".bp9Aid").get_attribute("innerHTML")

In [ ]:
# 별점
rating = float(driver.find_element(By.CSS_SELECTOR, 'div[aria-label*="별표 5개 만점에"]').get_attribute("aria-label").split()[3].replace("개를", ""))

In [ ]:
# 사용자 리뷰
user_review = driver.find_element(By.CSS_SELECTOR, ".h3YV2d").get_attribute("innerHTML")

In [ ]:
# 회사 응답
company_reply = driver.find_element(By.CSS_SELECTOR, ".ras4vb > div").get_attribute("innerHTML")

In [ ]:
from datetime import datetime, timedelta

In [ ]:
# 추출한 리뷰일을 날짜형 데이터로 변경
# datetime.srtptime(yyyy-mm-dd, "%Y-%m-%d") 날짜형 데이터 타입으로 변환
review_date = review_date.replace(" ", "").replace("년", "-").replace("월", "-").replace("일", "")
review_date = datetime.strptime(review_date, "%Y-%m-%d")

In [ ]:
review_date.date()

In [ ]:
today = datetime.today()
one_month_ago = today - timedelta(days=30)
one_month_ago.date()

In [ ]:
if review_date.date() < one_month_ago.date():
    break

# 코드 합쳐서 토스 리뷰 1달치 수집하기

In [4]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [5]:
import time
import pandas as pd
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [6]:
def to_date(review_date):
    # datetime.srtptime(yyyy-mm-dd, "%Y-%m-%d") 날짜형 데이터 타입으로 변환
    review_date = review_date.replace(" ", "").replace("년", "-").replace("월", "-").replace("일", "")
    review_date = datetime.strptime(review_date, "%Y-%m-%d")
    return review_date.date()

In [7]:
apps = dict(토스="viva.republica.toss", KB스타뱅킹="com.kbstar.kbbank", 하나원큐="com.kebhana.hanapush", 뱅크샐러드="com.rainist.banksalad2", 핀다="kr.co.finda.finda")

In [8]:
def app_review_extractor(app, days=7):
    # 웹브라우저를 실행 할 때 사용할 옵션 - 사람인 것처럼 정보 입력
    options = Options()

    # ✅ 브라우저가 꺼지지 않도록 유지
    options.add_experimental_option("detach", True)

    # ✅ 실제 사용자 환경과 유사한 기본 설정
    options.add_argument("--start-maximized")                # 최대화된 창으로 시작
    options.add_argument("--disable-blink-features=AutomationControlled")  # 자동화 탐지 회피
    options.add_argument("--disable-infobars")               # “Chrome이 자동 테스트 중” 문구 제거
    options.add_argument("--disable-extensions")             # 불필요한 확장 기능 제거
    options.add_argument("--disable-popup-blocking")         # 팝업 차단 비활성화 (사람처럼 행동)
    options.add_argument("--no-sandbox")                     # 보안 샌드박스 비활성화
    options.add_argument("--disable-dev-shm-usage")          # 리눅스 환경 안정성 향상

    # ✅ 실제 브라우저 User-Agent (사람의 브라우저처럼 보이도록)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/141.0.0.0 Safari/537.36"
    )

    # ✅ 시스템 언어 (정상적인 로케일 표현)
    options.add_argument("--lang=ko-KR")

    # ✅ 자동화 플래그 제거
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    # ✅ WebDriver 감지 우회용 스크립트 (선택적)
    prefs = {"credentials_enable_service": False, "profile.password_manager_enabled": False}
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    # 웹브라우저에서 url 열기
    driver.get(f"https://play.google.com/store/apps/details?id={app[1]}")

    # 요소가 실행 가능해질 때 까지 기다리기
    wait = WebDriverWait(driver, 10)

    # 평점 및 리뷰 옆의 -> 버튼이 활성화 되도록 스크롤을 1400px 아래로 내림
    driver.execute_script(f"window.scrollTo({0}, {1400})")
    time.sleep(2)

    # 평점 및 리뷰 옆의 -> 버튼 클릭
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[aria-label*="평점 및 리뷰 자세히 알아보기"]')))
    button.click()

    # 최신순으로 리뷰를 정렬하기 위해서 버튼 클릭
    wait.until(EC.element_to_be_clickable((By.ID, "sortBy_1"))).click()
    time.sleep(2)

    # 최신을 찾아 클릭
    wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'span[aria-label*="최신"]'))).click()
    time.sleep(2)

    # 스크롤을 내린 후 마지막 자료의 날짜를 찾아서 오늘 날짜부터 1주일 전 날짜와 비교하기

    # 리뷰가 담긴 창을 찾아서 Javascript로 10000px씩 아래로 스크롤
    today = datetime.today()
    end_date = today - timedelta(days=days)
    end_date = end_date.date()
    last_review_date = datetime.today().date()
    end = 1000
    len_list_old = len(driver.find_elements(By.CSS_SELECTOR, "div.RHo1pe"))
    while last_review_date > end_date:
        driver.execute_script(f"document.querySelector('.fysCi.Vk3ZVd').scrollTo(0, {end})")
        time.sleep(2)
        # 리뷰 목록 가져오기 div.RHo1pe
        review_list = driver.find_elements(By.CSS_SELECTOR, "div.RHo1pe")
        len_list_new = len(review_list)
        if len_list_old >= len_list_new:
            end += 10000
        else:
            len_list_old = len_list_new


        last_review_date = review_list[-1]
        last_review_date = last_review_date.find_element(By.CSS_SELECTOR, ".bp9Aid").get_attribute("innerHTML")
        last_review_date = to_date(last_review_date)

        print(len(review_list))
        print(last_review_date, end_date)


    print(len(review_list))
    print(end)

    # 리뷰 목록 가져오기 div.RHo1pe
    # review_list = driver.find_elements(By.CSS_SELECTOR, "div.RHo1pe")
    result = {}
    cols = ('리뷰일', '앱이름','별점', '사용자 리뷰','회사 응답')
    for review in review_list:

        # 리뷰일
        review_date = review.find_element(By.CSS_SELECTOR, ".bp9Aid").get_attribute("innerHTML")
        # 추출한 리뷰일을 날짜형 데이터로 변경
        review_date = to_date(review_date)

        # 별점
        rating = float(review.find_element(By.CSS_SELECTOR, 'div[aria-label*="별표 5개 만점에"]').get_attribute("aria-label").split()[3].replace("개를", ""))

        # 사용자 리뷰
        user_review = review.find_element(By.CSS_SELECTOR, ".h3YV2d").get_attribute("innerHTML")

        # 회사 응답
        try:
            company_reply = review.find_element(By.CSS_SELECTOR, ".ras4vb > div").get_attribute("innerHTML")
        except:
            company_reply = "회사 응답 없음"

        values = (review_date, app[0], rating, user_review, company_reply)
        for key, value in zip(cols, values):
            result.setdefault(key, []).append(value)
    driver.close()    
    df = pd.DataFrame(result)
    return df

In [ ]:
for app in apps.items():
    result = app_review_extractor(app)
    display(result)